<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Finding How The Data Is Distributed**


Estimated time needed: **30** minutes


In this lab, you will work with a cleaned dataset to perform Exploratory Data Analysis (EDA). You will examine the structure of the data, visualize key variables, and analyze trends related to developer experience, tools, job satisfaction, and other important aspects.


## Objectives


In this lab you will perform the following:


- Understand the structure of the dataset.

- Perform summary statistics and data visualization.

- Identify trends in developer experience, tools, job satisfaction, and other key variables.


### Install the required libraries


!pip install pandas
!pip install matplotlib
!pip install seaborn


### Step 1: Import Libraries and Load Data


- Import the `pandas`, `matplotlib.pyplot`, and `seaborn` libraries.


- You will begin with loading the dataset. You can use the pyfetch method if working on JupyterLite. Otherwise, you can use pandas' read_csv() function directly on their local machines or cloud environments.


In [ ]:
# Import necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the Stack Overflow survey dataset
data_url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv'
df = pd.read_csv(data_url)

# Display the first few rows of the dataset
df.head()


### Step 2: Examine the Structure of the Data


- Display the column names, data types, and summary information to understand the data structure.

- Objective: Gain insights into the dataset's shape and available variables.


In [ ]:
# Set pandas configuration options to stop truncation
pd.set_option('display.max_colwidth', None) # Shows full column names/values
pd.set_option('display.max_rows', None)     # Shows all rows
pd.set_option('display.max_columns', None)  # Shows all columns
pd.set_option('display.width', None)        # No wrapping of lines (optional)

# Optional: If you want to see the full text description of data types too
pd.set_option('display.expand_frame_repr', False) 

In [ ]:


## Write your code here
print("Column Names:", df.columns.tolist())
print("\nData Types:")
print(df.dtypes)


### Step 3: Handle Missing Data


- Identify missing values in the dataset.

- Impute or remove missing values as necessary to ensure data completeness.



In [ ]:
## Write your code here
import numpy as np

# Identify missing values first to track progress
missing_values = df.isnull().sum()

print("Missing Values Count:")
print(missing_values)

# --- IMPUTATION STEP (Robust Logic) ---

for col in df.columns:
    # Check if there are any missing values in this column
    if missing_values[col] > 0:
        print(f"\nProcessing column '{col}' with {missing_values[col]} missing values...")
        
        # Get the non-null values to determine their type and calculate statistics
        valid_data = df[col].dropna()
        
        # Determine if the data is numeric or not
        try:
            mean_val = valid_data.mean()
            is_numeric = True
            fill_value = mean_val  # Use Mean for numbers
            
            print(f"  -> Detected as Numeric. Filling with Mean: {fill_value}")
            
        except TypeError:
            is_numeric = False
            fill_value = "Not Applicable"  # Default string placeholder
            
        # Apply the imputation
        df[col] = df[col].fillna(fill_value)

# Final verification
print("\nMissing Values Count After Imputation:")
print(df.isnull().sum())


In [ ]:
# I want to see if there are any missing values in dataframe    
print(df.isnull().sum())

### Step 4: Analyze Key Columns


- Examine key columns such as `Employment`, `JobSat` (Job Satisfaction), and `YearsCodePro` (Professional Coding Experience).

- **Instruction**: Calculate the value counts for each column to understand the distribution of responses.



In [ ]:
## Write your code here
# Analyze key columns: Employment, JobSat, and YearsCodePro

print("Employment Value Counts:")
employment_counts = df['Employment'].value_counts()
print(employment_counts)

print("\nJob Satisfaction (JobSat) Value Counts (Top 10):")
job_sat_counts = df['JobSat'].value_counts().head(10) # Showing top 10 unique values
print(job_sat_counts)

# Corrected: Calling value_counts for the third column as well
print("\nYearsCodePro Value Counts:") 
years_code_pro_counts = df['YearsCodePro'].value_counts()
print(years_code_pro_counts)

### Step 5: Visualize Job Satisfaction (Focus on JobSat)


- Create a pie chart or KDE plot to visualize the distribution of `JobSat`.

- Provide an interpretation of the plot, highlighting key trends in job satisfaction.


In [ ]:
## Write your code here
# --- Visualization of Job Satisfaction (JobSat) ---

# Option A: Create a Pie Chart for Job Satisfaction
# This visualizes the proportion of each satisfaction score level.
plt.figure(figsize=(8, 6))
pie_data = df['JobSat'].value_counts() # Grouping by value counts
plt.pie(pie_data, labels=pie_data.index, autopct='%1.1f%%', startangle=90)
plt.title('Pie Chart: Distribution of Job Satisfaction Levels')
plt.axis('equal')  # Ensures pie is drawn as a circle
plt.show()

# Option B: Create a KDE Plot (Kernel Density Estimate)
# This shows the density distribution, useful for seeing if satisfaction is bimodal.
plt.figure(figsize=(8, 6))
sns.kdeplot(df['JobSat'], shade=True, fill=False, color='green', label="Distribution")
plt.title('KDE Plot: Job Satisfaction Distribution')
plt.xlabel("Job Satisfaction Score")
plt.ylabel("Density")
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

### Step 6: Programming Languages Analysis


- Compare the frequency of programming languages in `LanguageHaveWorkedWith` and `LanguageWantToWorkWith`.
  
- Visualize the overlap or differences using a Venn diagram or a grouped bar chart.


In [ ]:
## Write your code here
# ---- 1. Prepare language frequency counts ----

def get_language_counts(col_name):
    # Split comma-separated strings into individual languages
    lang_series = df[col_name].dropna().str.split(',')
    # Flatten list of lists into a single Series
    all_langs = [lang.strip() for sublist in lang_series for lang in sublist]
    return pd.Series(all_langs).value_counts()

# Count languages the respondents have worked with
worked_with_counts = get_language_counts('LanguageHaveWorkedWith')

# Count languages respondents want to work with
want_to_work_counts = get_language_counts('LanguageWantToWorkWith')

# ---- 2. Merge counts for a grouped bar chart ----
top_n = 10  # show top 10 languages

common_langs = worked_with_counts.head(top_n).index.union(want_to_work_counts.head(top_n).index)

merged_df = pd.DataFrame({
    'Worked With': worked_with_counts.loc[common_langs].reindex(common_langs, fill_value=0),
    'Want To Work': want_to_work_counts.loc[common_langs].reindex(common_langs, fill_value=0)
}).sort_values(by='Worked With', ascending=False)

# ---- 3. Plot grouped bar chart ----
merged_df.plot(kind='bar', figsize=(12, 6), width=0.8, color=['steelblue', 'orange'])
plt.title('Top Programming Languages: Worked With vs Want To Work')
plt.xlabel('Programming Language')
plt.ylabel('Number of Respondents')
plt.xticks(rotation=45, ha='right')
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

### Step 7: Analyze Remote Work Trends


- Visualize the distribution of RemoteWork by region using a grouped bar chart or heatmap.


In [ ]:
## Write your code here
# ---- 1. Prepare remote work counts per country (top 10) ----
remote_counts_country = df.groupby('Country')['RemoteWork'].value_counts().unstack(fill_value=0)

# Get top 10 countries by total remote‑work respondents
top_countries_remote = remote_counts_country.sum(axis=1).sort_values(ascending=False).head(10).index

# Filter the DataFrame to keep only those top countries
remote_top_df = remote_counts_country.loc[top_countries_remote]

# ---- 2. Plot grouped bar chart (descending order) ----
remote_top_df.plot(kind='bar', figsize=(12,6), width=0.8, color=['darkcyan','goldenrod'])
plt.title('Top Countries – Remote Work Preferences')
plt.xlabel('Country')
plt.ylabel('Number of Respondents')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### Step 8: Correlation between Job Satisfaction and Experience


- Analyze the correlation between overall job satisfaction (`JobSat`) and `YearsCodePro`.
  
- Calculate the Pearson or Spearman correlation coefficient.


In [ ]:
print("Unique values in 'YearsCodePro':")
print(df['YearsCodePro'].unique())

In [ ]:



# --- Step A: Handle Specific Invalid Strings ---
# Define the specific invalid strings mentioned in your request
invalid_strings = ['Not Applicable', 'Less than 1 year', 'More than 50 years']

# Create a temporary mapping to replace these with a placeholder (e.g., -99) 
# This helps us separate them from valid numbers if needed, but we can also just do this directly.
impute_map = {}
for s in invalid_strings:
    # We will store the "popular value" later, so for now let's assume we map to a placeholder 
    # or handle the logic below. 
    pass

# --- Step B: Find the "Popular Value" (Mode) among valid entries ---
# Filter the column to keep only rows that do NOT contain the invalid strings AND are not empty/NaN
valid_mask = df['YearsCodePro'].notna() & ~df['YearsCodePro'].isin(invalid_strings)

# Extract valid string values and convert to float immediately
try:
    # Attempt conversion on just the valid part to ensure no other non-numeric garbage exists (e.g. "NaN" text)
    clean_floats = df.loc[valid_mask, 'YearsCodePro'].astype(float)
    
    # Calculate Mode (Most Frequent Value). If there's a tie, .mode() returns all; we take the first.
    popular_val = float(clean_floats.mode()[0])
    
    print(f"Calculated Popular Value (Mode): {popular_val}")

except Exception as e:
    print(f"Error finding mode or converting valid strings: {e}")
    # Fallback if no numeric data found
    popular_val = 1.0 

# --- Step C: Create Final Imputation Dictionary ---
final_map = {}
for s in invalid_strings:
    final_map[s] = popular_val

# Also handle NaN/None values which might exist
final_map[None] = popular_val 
# Handle 'Not Applicable' explicitly if it wasn't caught by isin (e.g. if input was object type)
if isinstance(df['YearsCodePro'].iloc[0], str):
    # If the column contains strings, 'Not Applicable' might not be in invalid_strings list exactly or case-sensitive
    final_map['Not Applicable'] = popular_val

# --- Step D: Apply Imputation and Convert to Float ---
df['YearsCodePro'] = df['YearsCodePro'].replace(final_map).astype(float)

# Verification
print(f"\nColumn dtype after fix: {df['YearsCodePro'].dtype}")
print(f"Unique values remaining in column: {df['YearsCodePro'].unique()}")

if len(df['YearsCodePro'].unique()) == 1 and df['YearsCodePro'].iloc[0] != popular_val:
    print("Warning: All data might have been converted to the same value (unexpected).")
else:
    print(f"Success: Column is now Float64 with values distributed around {popular_val} where missing.")


In [ ]:

import numpy as np

# --- Step 1: Define Mapping for Invalid Strings ---
invalid_strings = ['Not Applicable', 'Less than 1 year', 'More than 50 years']

# Calculate the "Popular Value" (Mode) from existing valid numeric strings 
valid_data = df['YearsCodePro'].dropna()
valid_mask = ~valid_data.isin(invalid_strings) # Keep only valid numbers

if len(valid_data[valid_mask]) > 0:
    popular_val = float(pd.to_numeric(valid_data[valid_mask]).mode()[0])
else:
    popular_val = 1.0

impute_dict = {
    'Less than 1 year': 0.5,
    'More than 50 years': 51.0,
    'Not Applicable': popular_val
}

# --- Step 2: Apply Imputation and Convert to Float ---
df['YearsCodePro'] = df['YearsCodePro'].replace(impute_dict).astype(float)
print(f"Column 'YearsCodePro' cleaned. New dtype: {df['YearsCodePro'].dtype}")

# --- Step 3: Calculate Spearman Correlation (Robust Method) ---
col1 = 'JobSat'
col2 = 'YearsCodePro'

def spearman_correlation(x, y):
    """
    Calculates Spearman rank correlation using NumPy.
    This method avoids deprecated Pandas arguments like na_option='min'.
    """
    # Rank the data for x and y
    # We use a simple sort to determine ranks manually if needed, 
    # but pd.rank is safe here since we cleaned NaNs above.
    rank_x = pd.Series(x).rank(method='average')
    rank_y = pd.Series(y).rank(method='average')
    
    # Calculate Pearson correlation of ranks using NumPy
    cov_matrix = np.corrcoef(rank_x.values, rank_y.values)
    return cov_matrix[0, 1]

try:
    corr_coef = spearman_correlation(df[col1], df[col2])
    
    print(f"--- Result ---")
    print(f"Spearman Correlation Coefficient: {corr_coef:.4f}")
except Exception as e:
    # Fallback if data has issues (e.g., all values identical)
    print(f"Warning: Could not calculate correlation due to uniform data or error. Value: {e}")

### Step 9: Cross-tabulation Analysis (Employment vs. Education Level)


- Analyze the relationship between employment status (`Employment`) and education level (`EdLevel`).

- **Instruction**: Create a cross-tabulation using `pd.crosstab()` and visualize it with a stacked bar plot if possible.


In [ ]:
# --- Task 9: Cross-tabulation Analysis (Employment vs. Education Level) ---

print("=== Step 1: Create Full Cross-Tabulation ===\n")

# Create the full table using crosstab
cross_tab = pd.crosstab(df['Employment'], df['EdLevel'])

print(f"Full Table Shape: {cross_tab.shape}")

# --- Step 2: Identify Top Employment Types (Limit Rows) ===
# Calculate total respondents per employment type to rank them.
employment_totals = cross_tab.sum(axis=1) 

# Sort by count and pick the top N types (e.g., top 4 most common). 
# This ensures we only plot the most relevant rows while keeping all education levels.
top_n_employments = 4 
selected_employments = employment_totals.nlargest(top_n_employments).index.tolist()

print(f"Selected Employment Types: {selected_employments}")

# --- Step 3: Filter and Clean Data ===
# Filter the dataframe to keep only selected rows.
summary_plot_data = cross_tab.loc[selected_employments]

# Remove columns (Education Levels) that have zero counts across ALL selected employment types.
# This prevents "empty" bars in the graph for education levels with no respondents.
valid_columns_mask = summary_plot_data.sum(axis=0).gt(0) 
summary_plot_data_cleaned = summary_plot_data.loc[:, valid_columns_mask]

print(f"Columns removed (zero count): {len(summary_plot_data.columns) - len(valid_columns_mask)}")

# --- Step 4: Plotting with Explicit Legend Handling ===
if not summary_plot_data_cleaned.empty:
    # Create figure with adjusted size for better layout
    fig, ax = plt.subplots(figsize=(16, 8))
    
    # Plot stacked bars
    summary_plot_data_cleaned.plot(kind='bar', stacked=True, ax=ax, colormap='tab20')
    
    # Set titles and labels
    plt.title('Stacked Bar: Top Employment Types vs. Education Levels', fontsize=16, pad=20)
    plt.xlabel('Education Level (EdLevel)', fontsize=12)
    plt.ylabel('Number of Respondents', fontsize=12)

    # Add legend outside the plot area
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, summary_plot_data_cleaned.index, title='Employment Type', 
              loc='lower right', bbox_to_anchor=(1.0, 0.5), fontsize=14)
    
    # Handle X-axis rotation and spacing
    plt.xticks(rotation=45, ha='right')
    
    # Adjust layout to prevent clipping
    plt.tight_layout()
    plt.subplots_adjust(right=0.8)  # Make room for the legend

    plt.show()
else:
    print("Error: All data was zero or NaN.")
    
# --- Optional: Print Summary Table ===
print("\n--- Detailed Breakdown for Selected Employment Types (Cleaned) ---")
print(summary_plot_data_cleaned)

### Step 10: Export Cleaned Data


- Save the cleaned dataset to a new CSV file for further use or sharing.


In [ ]:
## Write your code here
# --- Step 10: Export Cleaned Data ---

# Save the cleaned dataset to a new CSV file
#df.to_csv('cleaned_survey_data.csv', index=False)

print("Dataset successfully saved as 'cleaned_survey_data.csv'")

### Summary:


In this lab, you practiced key skills in exploratory data analysis, including:


- Examining the structure and content of the Stack Overflow survey dataset to understand its variables and data types.

- Identifying and addressing missing data to ensure the dataset's quality and completeness.

- Summarizing and visualizing key variables such as job satisfaction, programming languages, and remote work trends.

- Analyzing relationships in the data using techniques like:
    - Comparing programming languages respondents have worked with versus those they want to work with.
      
    - Exploring remote work preferences by region.

- Investigating correlations between professional coding experience and job satisfaction.

- Performing cross-tabulations to analyze relationships between employment status and education levels.


## Authors:
Ayushi Jain


### Other Contributors:
Rav Ahuja
Lakshmi Holla
Malika


Copyright © IBM Corporation. All rights reserved.
